In [111]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pylab import rcParams
rcParams['figure.figsize'] = 15, 6

import openturns as ot
import openturns.viewer as otv
ot.Log.Show(ot.Log.NONE)

import functions_probabilistic as fp

In [112]:
def dF_tilde(x):
    # Parameters
    u10 = x[0]
    d = -x[1] + 5.3

    # Terms
    F_tilde = (g*F)/(u10**2)
    d_tilde = (g*d)/(u10**2)

    return F_tilde, d_tilde

def H_tilde(x):
    # Parameters
    H_inf = 0.14 #constant

    F_tilde, d_tilde = dF_tilde(x)

    # Terms
    H_tilde_1 = np.tanh(0.343 * d_tilde**1.14)
    H_tilde_2 = np.tanh((4.41 * (10**-4) * (F_tilde**0.79)) / H_tilde_1)

    # Final calculation
    H_tilde = H_inf * (H_tilde_1 * H_tilde_2)**0.572

    return H_tilde

def T_tilde(x):
    # Parameters
    T_inf = 7.69 #constant

    F_tilde, d_tilde = dF_tilde(x)

    # Terms
    T_tilde_1 = np.tanh(0.10 * (d_tilde**2.01))
    T_tilde_2 = np.tanh((2.77 * (10**-7) * (F_tilde**1.45)) / T_tilde_1)

    # Final calculation
    T_tilde = T_inf * ((T_tilde_1 * T_tilde_2)**0.187)
    
    return T_tilde

In [113]:
U10 = ot.WeibullMin(15, 2.2)
d = ot.Normal(-2, 0.3)
x = (U10, d)

In [114]:
#constants
q_max = 10
F = 21990
g = 9.81
Rc = 1.13
beta = 0

gamma_b = 0.6
gamma_f = 0.95 
gamma_s = 1 - 0.0033 * beta # gamma_beta
gamma_n = 1 # gamma_nu
gamma = gamma_b * gamma_f * gamma_n * gamma_s

tan_a = 1/3.3

global F 
global g
global Rc
global gamma
global gamma_b
global tan_a
global q_max


In [115]:
def LSF(x):
    # Parameters
    u = x[0]
    d = -x[1] + 5.3

    H_t = H_tilde(x)
    T_t = T_tilde(x)
    
    H = H_t * (u**2 / g)
    T = T_t * (u / g)
    L_0 = (g * T**2) / (2 * np.pi)
    xi = tan_a / np.sqrt(H / L_0)
    
    # Main calculation
    term_1 = ((0.026 * gamma_b * xi) / np.sqrt(tan_a))
    term_2 = np.exp(-((2.5 * Rc) / (xi * H * gamma))**1.3)
    term_d = np.sqrt(g * (H**3))
    q = term_1 * term_2 * term_d * 1000
    q_condition = q_max - q
    return [q_condition]


In [116]:
print(LSF((24.6, -2)))

[np.float64(0.17413490693961897)]


In [121]:
fp.input_OpenTurns(x, ("wind speed", "water depth"), LSF, 0)

In [118]:
result, x_star, u_star, pf_FORM, beta = fp.run_FORM_analysis()

The FORM analysis took 0.023 seconds
FORM result, pf = 0.0503
FORM result, beta = 1.642

The design point in the u space:  [1.63773,-0.111808]
The design point in the x space:  [24.6442,-2.03354]


In [119]:
alpha_ot, alpha, sens = fp.importance_factors(result)

--- FORM Importance Factors (alpha) ---


Importance factors, from OpenTURNs:
   0.995
   0.005

Importance factors, based on normal vector in U-space = 
   0.998
  -0.068
Note: this will be different from result.getImportanceFactors()
if there are resistance variables.

Sensitivity of Reliability Index to Multivariate Distribution

Distribution item number: 0
  Item name: wind speed

Distribution item number: 1
  Item name: water depth

Distribution item number: 2
  Item name: NormalCopula
    +0.000e+00 for parameter R_2_1_copula


In [120]:
it = 0
maxit = 1000
pf_FORM = 1.0
Rc = np.round(Rc-0.5, 1)
dx = 0.1
while pf_FORM > 2.7e-5:
    fp.input_OpenTurns(x, ("wind speed", "water depth"), LSF, 0)
    result, x_star, u_star, pf_FORM, beta = fp.run_FORM_analysis(printing = False)

    Rc += dx
    it += 1

    print(f"Iteration: {it:.2f}, pf_FORM: {pf_FORM:.6f} Rc: {Rc:.2f} m")


Iteration: 1.00, pf_FORM: 0.395340 Rc: 0.70 m
Iteration: 2.00, pf_FORM: 0.298659 Rc: 0.80 m
Iteration: 3.00, pf_FORM: 0.215592 Rc: 0.90 m
Iteration: 4.00, pf_FORM: 0.148078 Rc: 1.00 m
Iteration: 5.00, pf_FORM: 0.096318 Rc: 1.10 m
Iteration: 6.00, pf_FORM: 0.059033 Rc: 1.20 m
Iteration: 7.00, pf_FORM: 0.033912 Rc: 1.30 m
Iteration: 8.00, pf_FORM: 0.018159 Rc: 1.40 m
Iteration: 9.00, pf_FORM: 0.009014 Rc: 1.50 m
Iteration: 10.00, pf_FORM: 0.004126 Rc: 1.60 m
Iteration: 11.00, pf_FORM: 0.001732 Rc: 1.70 m
Iteration: 12.00, pf_FORM: 0.000663 Rc: 1.80 m
Iteration: 13.00, pf_FORM: 0.000231 Rc: 1.90 m
Iteration: 14.00, pf_FORM: 0.000073 Rc: 2.00 m
Iteration: 15.00, pf_FORM: 0.000021 Rc: 2.10 m
